In [3]:
!pip install pypdf2 wikipedia nltk requests beautifulsoup4 scikit-learn

In [4]:
import re
import json
import nltk
import requests
import PyPDF2
import wikipedia

from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

print("All libraries imported successfully.")

All libraries imported successfully.


[nltk_data] Downloading package stopwords to C:\Users\Ishaan
[nltk_data]     Kapil\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
def clean_text(text):
    """
    General text cleaner for TF-IDF vectorization.
    - Lowercases, removes URLs/emails/special chars, removes stopwords.
    """
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split() if w not in stop_words]
    return ' '.join(tokens)


def clean_pdf_text(text):
    """
    PDF-specific cleaner:
    - Strips boilerplate footers, URLs, emails, author credits.
    - Fixes broken spacing from PyPDF2 extraction.
    """
    # Cut boilerplate from 'For more information go to:' onward
    text = re.split(r'For more information go to:', text, flags=re.IGNORECASE)[0]
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    # Fix single-char spacing artifacts from PDF extraction
    text = re.sub(r'(\b[A-Za-z])\s+(?=[A-Za-z]\b)', r'\1', text)
    text = re.sub(r'Written by.*', '', text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r'contact us.*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Reference:.*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


print("Text cleaning functions defined.")

Text cleaning functions defined.


In [6]:
def get_blog_data(url):
    """
    Scrapes paragraph text from a blog URL.
    Uses a browser User-Agent header to avoid 403 errors.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            paragraphs = soup.find_all('p')
            text = ' '.join(p.get_text(separator=' ') for p in paragraphs)
            return text
        else:
            return f"Failed to retrieve content. Status code: {response.status_code}"
    except Exception as e:
        return f"Error fetching blog: {e}"


blog_url = "https://iismworld.com/blog/success-story-of-indian-premier-league-ipl/"
blog_data = get_blog_data(blog_url)

print(f"Blog data collected. Length: {len(blog_data)} characters")
print("Preview:", blog_data, "...")

Blog data collected. Length: 6785 characters
Preview: Introduction: Millions of fans, crazy excitement, nail-biting suspense, fanaticism, and 1 IPL -India’s biggest ‘Cricketainment, perhaps one of the world’s too. A 18-year of phenomenal journey, and a massive success in the world of sports business. In 2008, from 8 franchises to 10 teams in 2025, Indian Premier League saw media rights commanding a staggering $6.2 billion with Mumbai Indians and Chennai Super Kings rivaling some of the biggest clubs in European football and NFL. IPL is revolutionary not only in the business of cricket but also as a global sporting powerhouse. As we celebrate the IPL like any other festival in India, let us reflect on how a startup league has become a massive success. IPL was a bold experiment that aimed to blend cricket and entertainment, creating an extravaganza for cricket fans. Though the concept of city-based rivalries, franchise models, and 20-over matches saw success globally, but it had not been

In [7]:
def get_wiki_data(topic):
    try:
        return wikipedia.summary(topic, sentences=30)
    except wikipedia.exceptions.DisambiguationError as e:
        try:
            return wikipedia.summary(e.options[0], sentences=30)
        except Exception:
            return f"Disambiguation error. Options: {e.options[:5]}"
    except wikipedia.exceptions.HTTPTimeoutError:
        return "Error: Timeout occurred while fetching Wikipedia data."
    except Exception as e:
        return f"Error: {e}"


wiki_data = get_wiki_data("Cricket")

print(f"Wikipedia data collected. Length: {len(wiki_data)} characters")
print("Preview:", wiki_data[:400], "...")

Wikipedia data collected. Length: 1950 characters
Preview: Cricket is a bat-and-ball game that is played between two teams of eleven players on a field, at the centre of which is a 22-yard (20-metre; 66-foot) pitch with a wicket at each end, each comprising two bails (small sticks) balanced on three stumps. Two players from the batting team, the striker and nonstriker, stand in front of either wicket holding bats, while one player from the fielding team,  ...


In [8]:
def get_pdf_data(pdf_path):
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    cleaned = clean_pdf_text(extracted)
                    text += cleaned + " "
        return text.strip()
    except FileNotFoundError:
        return f"Error: '{pdf_path}' not found. Place intro.pdf in the same directory."
    except Exception as e:
        return f"Error reading PDF: {e}"
pdf_data = get_pdf_data("intro.pdf")
print(f"PDF data collected. Length: {len(pdf_data)} characters")
print("Preview:", pdf_data[:400], "...")

PDF data collected. Length: 3173 characters
Preview: The Game of Cricket Batsmen Bowler Wickets Umpire Crease Cricket field: The rectangular patch in the center is called the pitch. Pitch A typical Cricket game has 11 players in each team. It is fundamentally very similar to baseball. It is played with a bat and a ball. The center of the field is a rectangular area of 22 meters called a pitch. In cricket there are only two bases (called creases) on  ...


In [9]:
raw_corpus = [
    {"source": "Wikipedia", "topic": "Cricket",                 "content": wiki_data},
    {"source": "Blog",      "topic": "IPL Success Story",                     "content": blog_data},
    {"source": "PDF",       "topic": "Cricket Rules and Basics", "content": pdf_data}
]
with open("cricket_corpus.json", "w", encoding="utf-8") as f:
    json.dump(raw_corpus, f, indent=4, ensure_ascii=False)
print("JSON corpus saved as 'cricket_corpus.json'")
print()
for item in raw_corpus:
    print(f"  Source: {item['source']:12s} | Topic: {item['topic']:26s} | Chars: {len(item['content'])}")

JSON corpus saved as 'cricket_corpus.json'

  Source: Wikipedia    | Topic: Cricket                    | Chars: 1950
  Source: Blog         | Topic: IPL Success Story          | Chars: 6785
  Source: PDF          | Topic: Cricket Rules and Basics   | Chars: 3173


In [10]:
def is_question_heading(sentence):
    s = sentence.strip()
    return s.endswith('?') and len(s) < 80


def split_into_sentences(text):
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 30]
    return sentences


def build_chunks(raw_corpus):
    chunks = []
    skipped = 0

    for item in raw_corpus:
        source   = item['source']
        topic    = item['topic']
        sentences = split_into_sentences(item['content'])

        for i in range(len(sentences)):
            s1 = sentences[i]

            # === SKIP standalone question headings ===
            if is_question_heading(s1):
                skipped += 1
                continue

            if len(s1) < 60:
                skipped += 1
                continue

            chunks.append({'source': source, 'topic': topic, 'content': s1})

            if i + 1 < len(sentences):
                s2 = sentences[i + 1]
                if not is_question_heading(s2):
                    combined2 = s1 + ' ' + s2
                    chunks.append({'source': source, 'topic': topic, 'content': combined2})

            if i + 2 < len(sentences):
                s2 = sentences[i + 1]
                s3 = sentences[i + 2]
                if not is_question_heading(s2) and not is_question_heading(s3):
                    combined3 = s1 + ' ' + s2 + ' ' + s3
                    chunks.append({'source': source, 'topic': topic, 'content': combined3})

    return chunks, skipped


chunks, skipped_count = build_chunks(raw_corpus)
print(f"Chunking complete.")
print(f"  Total chunks created : {len(chunks)}")
print(f"  Question headings skipped: {skipped_count}")
print()
print("Sample chunk (index 5):")
print(json.dumps(chunks[5], indent=2))

Chunking complete.
  Total chunks created : 225
  Question headings skipped: 8

Sample chunk (index 5):
{
  "source": "Wikipedia",
  "topic": "Cricket",
  "content": "Two players from the batting team, the striker and nonstriker, stand in front of either wicket holding bats, while one player from the fielding team, the bowler, bowls the ball toward the striker's wicket from the opposite end of the pitch. The striker's goal is to hit the bowled ball with the bat and then switch places with the nonstriker, with the batting team scoring one run for each of these swaps. Runs are also scored when the ball reaches the boundary of the field or when the ball is bowled illegally."
}


In [11]:
def prepare_chunk_texts(chunks):
    return [clean_text(f"{ch['topic']} {ch['content']}") for ch in chunks]
chunk_docs = prepare_chunk_texts(chunks)

word_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=1)
word_matrix     = word_vectorizer.fit_transform(chunk_docs)

char_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=1)
char_matrix     = char_vectorizer.fit_transform(chunk_docs)

print(f"Word TF-IDF matrix shape: {word_matrix.shape}")
print(f"Char TF-IDF matrix shape: {char_matrix.shape}")

Word TF-IDF matrix shape: (225, 1661)
Char TF-IDF matrix shape: (225, 6025)


In [12]:
last_topic_hint = ""
def expand_followup_query(query, last_topic_hint):
    q = query.strip()
    vague_words = {'it', 'this', 'that', 'they', 'them', 'its', 'their', 'format', 'winner', 'won'}
    if any(w in clean_text(q).split() for w in vague_words):
        if last_topic_hint:
            q = q + ' ' + last_topic_hint
    return q


def rank_chunks(query, chunks, top_n=5):
    query_clean = clean_text(query)
    if not query_clean.strip():
        return []

    q_word = word_vectorizer.transform([query_clean])
    q_char = char_vectorizer.transform([query_clean])

    word_scores = cosine_similarity(q_word, word_matrix)[0]
    char_scores = cosine_similarity(q_char, char_matrix)[0]
    final_scores = 0.75 * word_scores + 0.25 * char_scores
    query_words = set(query_clean.split())
    for i, ch in enumerate(chunks):
        text_clean = clean_text(ch['content'])
        text_words = set(text_clean.split())
        overlap    = len(query_words & text_words)          
        boost      = min(0.02 * overlap, 0.10)              
        final_scores[i] += boost

    ranked_indices = final_scores.argsort()[::-1]

    results = []
    for idx in ranked_indices[:top_n * 3]:                  
        ch = chunks[idx]

        # === HARD FILTER: skip question headings in results ===
        if is_question_heading(ch['content']):
            continue

        if final_scores[idx] < 0.10:                     
            continue

        results.append((ch, float(final_scores[idx])))
        if len(results) >= top_n:
            break

    return results


def search(query):
    global last_topic_hint

    expanded_query = expand_followup_query(query, last_topic_hint)
    results = rank_chunks(expanded_query, chunks, top_n=5)

    if not results:
        return "Sorry, I could not find a relevant answer in the corpus."

    best_chunk, best_score = results[0]
    last_topic_hint = best_chunk['topic']

    return (
        f"Source     : {best_chunk['source']}\n"
        f"Topic      : {best_chunk['topic']}\n"
        f"Answer     : {best_chunk['content']}\n"
        f"Similarity : {best_score:.2f}"
    )


print("Search and retrieval functions defined.")

Search and retrieval functions defined.


In [13]:
def chatbot():
    print("=" * 60)
    print(" Cricket & IPL Chatbot")
    print(" Sources: Wikipedia | Betway Blog | PDF (intro.pdf)")
    print(" Type 'exit' to quit.")
    print("=" * 60)
    print()

    greetings = {'hi', 'hello', 'hey', 'hiya', 'howdy'}

    while True:
        query = input("You: ").strip()
        if not query:
            continue
        if query.lower() in {'exit', 'quit', 'bye'}:
            print("Bot: Goodbye!")
            break
        if query.lower() in greetings:
            print("Bot: Hello! Ask me anything about Cricket or the IPL.\n")
            continue
        print()
        print("Bot:", search(query))
        print()


chatbot()

 Cricket & IPL Chatbot
 Sources: Wikipedia | Betway Blog | PDF (intro.pdf)
 Type 'exit' to quit.



You:  How many teams are in the IPL?



Bot: Source     : Blog
Topic      : IPL Success Story
Answer     : Key Takeaways: The IPL was a true global competition, involving international players from teams like Australia, South Africa, New Zealand, West Indies and many more.
Similarity : 0.24



You:  What role do international players play in IPL?



Bot: Source     : Blog
Topic      : IPL Success Story
Answer     : Key Takeaways: The IPL was a true global competition, involving international players from teams like Australia, South Africa, New Zealand, West Indies and many more. This blend of domestic and international players amped up the standard of the league and attracted fans from around the world.
Similarity : 0.41



...


Bot: Source     : PDF
Topic      : Cricket Rules and Basics
Answer     : The Game of Cricket Batsmen Bowler Wickets Umpire Crease Cricket field: The rectangular patch in the center is called the pitch. Pitch A typical Cricket game has 11 players in each team.
Similarity : 0.38



You:  Tell about several type of cricket games?



Bot: Source     : PDF
Topic      : Cricket Rules and Basics
Answer     : There are several types of cricket games, but we would talk about one-day game in this discussion.
Similarity : 0.51



You:  EXIT


Bot: Goodbye!


In [14]:
print('=' * 60)
print(' CORPUS CREATION PIPELINE — SUMMARY')
print('=' * 60)
print(f"""
Domain        : Cricket & IPL

Sources       :
  1. Wikipedia      — wikipedia.summary('Cricket', sentences=30)
  2. Blog           — requests + BeautifulSoup scraping of <p> tags
       URL: https://iismworld.com/blog/success-story-of-indian-premier-league-ipl/
  3. PDF (intro.pdf)— PyPDF2.PdfReader, page-by-page extraction

Preprocessing :
  - Lowercase, URL/email/special char removal
  - PDF: boilerplate footer, author credits stripped
  - PyPDF2 broken char-spacing artifact fix
  - NLTK English stopword removal

Retrieval     : TF-IDF (Word + Char) + Cosine Similarity

Improvements  :
  1. normalize_query()     — expands IPL, LBW, ICC, KKR, CSK, T20
                             before TF-IDF vectorization (now ACTIVE
                             in search(), called every query)
  2. Smart Chunking        — 1/2/3-sentence overlapping windows;
                             question headings filtered;
                             PDF title/header line filtered
  3. Threshold = 0.20      — raised from 0.10, cuts weak answers
  4. expand_followup_query — topic memory for vague follow-ups

Corpus Stats  :
  Total sources    : {len(raw_corpus)}
  Total chunks     : {len(chunks)}
  Headings skipped : {skipped_count}
  Word TF-IDF shape: {word_matrix.shape}
  Char TF-IDF shape: {char_matrix.shape}
  Output file      : cricket_corpus.json
""")

with open('cricket_corpus.json', 'r', encoding='utf-8') as f:
    loaded = json.load(f)
print('JSON corpus verified:')
for item in loaded:
    print(f"  {item['source']:12s} | {item['topic']:48s} | {len(item['content'])} chars")

 CORPUS CREATION PIPELINE — SUMMARY

Domain        : Cricket & IPL

Sources       :
  1. Wikipedia      — wikipedia.summary('Cricket', sentences=30)
  2. Blog           — requests + BeautifulSoup scraping of <p> tags
       URL: https://iismworld.com/blog/success-story-of-indian-premier-league-ipl/
  3. PDF (intro.pdf)— PyPDF2.PdfReader, page-by-page extraction

Preprocessing :
  - Lowercase, URL/email/special char removal
  - PDF: boilerplate footer, author credits stripped
  - PyPDF2 broken char-spacing artifact fix
  - NLTK English stopword removal

Retrieval     : TF-IDF (Word + Char) + Cosine Similarity

Improvements  :
  1. normalize_query()     — expands IPL, LBW, ICC, KKR, CSK, T20
                             before TF-IDF vectorization (now ACTIVE
                             in search(), called every query)
  2. Smart Chunking        — 1/2/3-sentence overlapping windows;
                             question headings filtered;
                             PDF title/header li